# 03 — PPE Integration (Full Scale)

Builds all PPE-derived plant-side representations from the full
opportunity surface (6,697 parquet files):

1. **f_curves** — normalized 52-week flowering probability per species
   (used for Δ computation in ANTHEIA-Scalar)
2. **V_δ (4D)** — PCA 4D spatiotemporal plant embedding
   (used in ANTHEIA-4D)
3. **V_δ (15D)** — PCA 15D spatiotemporal plant embedding
   (used in ANTHEIA-15D)
4. **PMf / Vf_prob** — continuous PPE probability replacing binary Vf
   (used in ANTHEIA-PMf)

**Critical:** f_curves must be built from the full PPE opportunity surface
parquet files (`part_*.parquet`), NOT from `flowering_curves_used.parquet`
which covers only 50 species from the exploratory phase.

**Outputs:**
- `f_curves_ppe.csv` — (6,697 species × 52 weeks)
- `stage5_Vdelta_ppe.csv` — V_δ 4D (6,697 species × 4)
- `stage5_Vdelta_15d.csv` — V_δ 15D (6,697 species × 15)
- `stage5_Vf_prob.csv` — PMf Vf_prob (6,466 species × 15)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from pathlib import Path
import glob
import gc

BASE     = Path("/scratch/ariana.l")
PPE_DIR  = BASE / "ppe-outputs" / "opportunity_surface"
OLD_S4   = BASE / "Stage 4 Link Prediction Model"
NEW_S4   = BASE / "New Stage 4 Link Prediction Model"
STAGE5   = BASE / "Stage 5 PPE Representation Study"
STAGE5.mkdir(parents=True, exist_ok=True)

F_PATH   = OLD_S4 / "stage4_F_existence_phenofield.csv"

print("Paths OK")
print(f"  PPE_DIR : {PPE_DIR}")
files = sorted(glob.glob(str(PPE_DIR / "part_*.parquet")))
print(f"  PPE files: {len(files)}")
# Expected: 6,697

In [ ]:
# Build f_curves: normalized 52-week mean flowering probability per species
# Stream all parquet files, one species per file
print("Building f_curves from PPE opportunity surface...")

f_curves_dict = {}
for i, fpath in enumerate(files):
    df = pd.read_parquet(fpath, columns=["species", "week", "norm"])
    species = df["species"].iloc[0]
    weekly_mean = df.groupby("week")["norm"].mean().reindex(range(52), fill_value=0)
    total = weekly_mean.sum()
    if total > 0:
        weekly_mean = weekly_mean / total
    f_curves_dict[species] = weekly_mean.values
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(files)} files processed...")

f_curves_df = pd.DataFrame.from_dict(
    f_curves_dict, orient="index",
    columns=[f"w{i}" for i in range(52)]
)
print(f"  f_curves shape: {f_curves_df.shape}")
# Expected: (6697, 52)

f_curves_df.to_csv(NEW_S4 / "f_curves_ppe.csv")
print(f"  Saved → f_curves_ppe.csv")

In [ ]:
# Build spatiotemporal plant embedding V_δ
# For each plant species, load all (bin × week) opportunity values,
# flatten to a vector, then PCA compress.
#
# The opportunity surface has shape (n_bins × 52) per species.
# We flatten this to a single vector and PCA across species.

print("Building spatiotemporal opportunity surface matrix...")

# Load all files, pivot to (species × bins_weeks) matrix
surface_rows = {}
for i, fpath in enumerate(files):
    df = pd.read_parquet(fpath, columns=["species", "lat", "lon", "week", "norm"])
    species = df["species"].iloc[0]
    # Create bin label
    df["bin"] = (np.floor(df["lat"] / 0.5) * 0.5).round(1).astype(str) + "_" + \
                (np.floor(df["lon"] / 0.5) * 0.5).round(1).astype(str)
    pivot = df.groupby(["bin", "week"])["norm"].mean().unstack(fill_value=0)
    surface_rows[species] = pivot.values.flatten()
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(files)} files processed...")

print(f"  Surface matrix: {len(surface_rows)} species")

# Convert to DataFrame — rows may have different lengths if bins vary
# Use the shortest row length for safety
min_len = min(len(v) for v in surface_rows.values())
surface_df = pd.DataFrame(
    {sp: v[:min_len] for sp, v in surface_rows.items()}
).T
print(f"  Surface shape: {surface_df.shape}")

del surface_rows
gc.collect()

In [ ]:
# PCA 4D → V_δ (4D) for ANTHEIA-4D
print("Fitting PCA 4D on opportunity surface...")
pca_4d = PCA(n_components=4, svd_solver='randomized', random_state=42)
Vd4_arr = pca_4d.fit_transform(surface_df.values)
print(f"  Variance explained (4D): {pca_4d.explained_variance_ratio_.sum():.4f}")

Vd4_df = pd.DataFrame(Vd4_arr, index=surface_df.index,
                       columns=[f"PC{i+1}" for i in range(4)])
Vd4_df.to_csv(STAGE5 / "stage5_Vdelta_ppe.csv")
print(f"  Saved → stage5_Vdelta_ppe.csv")

In [ ]:
# PCA 15D → V_δ (15D) for ANTHEIA-15D
print("Fitting PCA 15D on opportunity surface...")
pca_15d = PCA(n_components=15, svd_solver='randomized', random_state=42)
Vd15_arr = pca_15d.fit_transform(surface_df.values)
print(f"  Variance explained (15D): {pca_15d.explained_variance_ratio_.sum():.4f}")
# Expected: ~0.486

Vd15_df = pd.DataFrame(Vd15_arr, index=surface_df.index,
                        columns=[f"PC{i+1}" for i in range(15)])
Vd15_df.to_csv(STAGE5 / "stage5_Vdelta_15d.csv")
print(f"  Saved → stage5_Vdelta_15d.csv")

del surface_df, Vd4_arr, Vd15_arr
gc.collect()

In [ ]:
# Build PMf / Vf_prob for ANTHEIA-PMf
# Replace binary F matrix with continuous PPE flowering probability
# For each plant, for each bin, mean norm across all 52 weeks → probability of flowering
# Then PCA 15D across species

print("Building PMf (continuous plant existence matrix)...")

# Load F to get the canonical bin set
F_df = pd.read_csv(F_PATH, index_col=0)
common_bins = list(F_df.columns)
common_bins_set = set(common_bins)

pmf_rows = {}
for i, fpath in enumerate(files):
    df = pd.read_parquet(fpath, columns=["species", "lat", "lon", "norm"])
    species = df["species"].iloc[0]
    if species not in set(F_df.index):
        continue
    df["bin"] = (np.floor(df["lat"] / 0.5) * 0.5).round(1).astype(str) + "_" + \
                (np.floor(df["lon"] / 0.5) * 0.5).round(1).astype(str)
    df = df[df["bin"].isin(common_bins_set)]
    bin_means = df.groupby("bin")["norm"].mean()
    row = pd.Series(0.0, index=common_bins)
    row.update(bin_means)
    pmf_rows[species] = row.values

print(f"  PMf rows: {len(pmf_rows)} species")

PMf_df = pd.DataFrame.from_dict(pmf_rows, orient="index", columns=common_bins)

# PCA 15D on PMf → Vf_prob
pca_pmf = PCA(n_components=15, svd_solver='randomized', random_state=42)
Vfp_arr = pca_pmf.fit_transform(PMf_df.values)
print(f"  PMf PCA variance explained: {pca_pmf.explained_variance_ratio_.sum():.4f}")

Vfp_df = pd.DataFrame(Vfp_arr, index=PMf_df.index,
                       columns=[f"PC{i+1}" for i in range(15)])
Vfp_df.to_csv(STAGE5 / "stage5_Vf_prob.csv")
print(f"  Saved → stage5_Vf_prob.csv")
print(f"  Vf_prob shape: {Vfp_df.shape}")

In [ ]:
# Summary
print("PPE integration complete.")
print(f"  f_curves  : {f_curves_df.shape}  → used for Δ in ANTHEIA-Scalar")
print(f"  V_δ 4D    : {Vd4_df.shape}    → used in ANTHEIA-4D")
print(f"  V_δ 15D   : {Vd15_df.shape}   → used in ANTHEIA-15D")
print(f"  Vf_prob   : {Vfp_df.shape}    → used in ANTHEIA-PMf")